In [1]:
def find_max_precip_location_new(cube, start_idx, stop_idx, x_offset=0, y_offset=0, mask_2d=None):
    
    # Slice the event window first — much smaller than full year
    #cube_sliced = cube[start_idx:stop_idx,:,:]
    data = np.array(cube.data)
    data[data >= 1e19] = np.nan

    # Then apply mask only to this small slice
    if mask_2d is not None:
        data = np.where(mask_2d, data, np.nan)

    flat_index = np.nanargmax(data)
    # print(f"Max precip: {np.nanmax(data)}")
    t_local, y_idx, x_idx = np.unravel_index(flat_index, data.shape)

    return {
        'max_precip':   float(data[t_local, y_idx, x_idx]),
        't_global':     int(t_local + start_idx),
        't_local':      int(t_local),
        'x_idx':        int(x_idx),
        'y_idx':        int(y_idx),
        'x_idx_global': int(x_idx + x_offset),
        'y_idx_global': int(y_idx + y_offset),
        'x_coord':      cube.coord('projection_x_coordinate').points[x_idx],
        'y_coord':      cube.coord('projection_y_coordinate').points[y_idx],
    }


def find_temporal_profile_new(cube, details, plot):
    # Extract event time series
    values = cube.data
    times = cube.coord('yyyymmddhh').points.astype(int)

    metrics = compute_temporal_metrics(values)
    
    # Convert to datetime
    dt = pd.to_datetime(times.astype(str), format="%Y%m%d%H")
    
    #temp_profile_dict = {'total_acc': values.sum(), 'times':dt, 'values':values}
    temp_profile_dict = {**metrics,'times': dt,'values': values}
    
    # Plot
    if plot == True:
        plt.figure(figsize=(8,4))
        plt.plot(dt, values, color='black')

        # Mark peak timestep
        rainfall_peak_time = cube.coord('yyyymmddhh').points[details['t_local']]
        peak_dt = pd.to_datetime(str(int(rainfall_peak_time)), format="%Y%m%d%H")
        plt.axvline(peak_dt, linestyle='--', color = 'red', label='Peak timestep')

        # Format ticks
        plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%H'))

        plt.xlabel("Date")
        plt.ylabel("Precipitation intensity (mm/hr)")
        plt.title("Rainfall at peak grid cell during event")
        plt.legend()

        plt.tight_layout()
        plt.show()

    return temp_profile_dict

In [2]:
import numpy as np
import pandas as pd
import cftime
import geopandas as gpd
import re
import time
import gc
import os
import shapely
from concurrent.futures import ProcessPoolExecutor, as_completed
import subprocess
import getpass
import signal
import sys
import logging
import math

from functions import *
from functions_stage2 import get_rainfall_cube_subsection

def setup_worker_logger(catchment_num):
    """Each worker logs to its own file so output never interleaves."""
    log_path = os.path.join(OUT_DIR, f"logs/catchment_{catchment_num}.log")
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    
    logger = logging.getLogger(f"catchment_{catchment_num}")
    logger.setLevel(logging.INFO)
    
    # Clear any existing handlers (important if function is called multiple times)
    logger.handlers.clear()
    
    fh = logging.FileHandler(log_path, mode='w')
    fh.setFormatter(logging.Formatter('%(asctime)s %(message)s', datefmt='%H:%M:%S'))
    logger.addHandler(fh)
    
    return logger

# ── Config ────────────────────────────────────────────────────────────────────
MOLLY_DIR_FF     = "/scratch/hydro4/users/kv25483/FutureFlood/"
RAINFALL_CSV_DIR = "/scratch/hydro4/users/la17355/FUTURE-FLOOD/UKCP_rainfall_events/fixed_threshold_30mm_with_volume/"
OUT_DIR          = "/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/"

from config import CATCHMENT_LOOKUP_DICT, OUT_DIR, CATCHMENTS, ENSEMBLE_MEMBERS # MOLLY_DIR_FF, RAINFALL_CSV_DIR, , ENSEMBLE_MEMBERS,  

## ------------------------------------------------------------ ###
# Define which catchments to run
## ------------------------------------------------------------ ###
all_catchments = set(CATCHMENT_LOOKUP_DICT.keys())
files = os.listdir(OUT_DIR)
completed_catchments = {re.search(r'Catchment_(.+)', f).group(1) for f in files if re.search(r'Catchment_(.+)', f)}
catchments_to_run = (all_catchments - completed_catchments) - {105} - {14}

catchments_with_flood_output = []
## Loop through all catchments
for catchment_num in all_catchments:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails/Catchment_{catchment_num}/{catchment_name}.pkl"
    flood_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/Catchment_{catchment_num}/Ens01_{catchment_num}/10cm/flooded_area_5km_total_Ens01_{catchment_num}_10cm.nc"
        
    if os.path.isfile(flood_fp):
        catchments_with_flood_output.append(catchment_num)

# ── Check which (catchment, ens) combinations still need processing ───────────
# Assumes outputs are saved as: OUT_DIR/Catchment_{num}/Ens_{ens}.pkl
# If your output structure is different, adjust the path accordingly.

missing = {}  # {catchment_num: [list of missing ens]}
to_prioritise ={}

for catchment_num in catchments_with_flood_output:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]

    # Load just enough to know which ens members exist for this catchment
    all_ens = ['01', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '15']

    overall_fp = out_fp = f"{OUT_DIR}/Catchment_{catchment_num}/{catchment_name}.pkl"
    if os.path.isfile(overall_fp):
        print(f"Catchment {catchment_num} ({catchment_name}) is complete")
        pass
    else:
    
        missing_ens = []
        for ens in sorted(all_ens):
            out_fp = f"{OUT_DIR}/Catchment_{catchment_num}/{catchment_name}_EM{ens}.pkl"
            if not os.path.isfile(out_fp):
                missing_ens.append(ens)

        if missing_ens:
            missing[catchment_num] = missing_ens
            print(f"Catchment {catchment_num} ({catchment_name}): missing ens {missing_ens}")
        else:
            print(f"Catchment {catchment_num} ({catchment_name}): complete")

# Summary
total_missing = sum(len(v) for v in missing.values())
print(f"\n{len(missing)} catchments incomplete, {total_missing} (catchment, ens) jobs remaining")        
        
# Sort catchments by number of missing ens (ascending) so we tackle the
# nearly-complete ones first — minimises total remaining work to get results.
sorted_missing = sorted(missing.items(), key=lambda x: len(x[1]))

print("\nCatchments sorted by missing ens (least first):")
print(f"{'Catchment':<15} {'Name':<30} {'Missing ens':<15} {'Ens list'}")
print("-" * 80)
for catchment_num, missing_ens in sorted_missing:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    print(f"{catchment_num:<15} {catchment_name:<30} {len(missing_ens):<15} {missing_ens}")

# Also gives you a clean list to feed directly into your processing loop
catchments_to_run_sorted = [catchment_num for catchment_num, _ in sorted_missing]

# ── Main processing function ──────────────────────────────────────────────────
def process_catchment(catchment_num, verbose=False):
    """
    verbose=True  → prints directly to console/notebook output
    verbose=False → writes to log file only (for use with ProcessPoolExecutor)
    """

    # ── Set up output routing ─────────────────────────────────────────────────
    # When running in parallel, verbose=False routes all output to a per-catchment
    # log file so workers don't interleave. When debugging in a notebook,
    # verbose=True prints directly so you see output immediately.
    log = setup_worker_logger(catchment_num)

    def emit(msg):
        """Single call routes to print or log depending on verbose flag."""
        if verbose:
            print(msg)
        else:
            log.info(msg)

    catchment_name  = CATCHMENT_LOOKUP_DICT[str(catchment_num)]
    emit(f"Starting — {catchment_name}")
    boundary_gdf    = CATCHMENTS[CATCHMENTS['HA_NUM'] == str(catchment_num)]
    _CATCHMENT_POLY = boundary_gdf.geometry.iloc[0]
    results         = []

    for ens_num in ENSEMBLE_MEMBERS:
        out_fp = f"{OUT_DIR}/Catchment_{catchment_num}/{catchment_name}_EM{ens_num}.pkl"
        if os.path.isfile(out_fp):
            emit(f"EM{ens_num}: already exists, skipping")
            continue

        results_this_ens = []
        rainfall_events  = pd.read_csv(RAINFALL_CSV_DIR + f"{catchment_name}_{ens_num}_full_events_with_event_nums.csv")
        rainfall_events['event_num'] = range(1, len(rainfall_events) + 1)
        rainfall_cube_dir = f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/{ens_num}/"
        emit(f"EM{ens_num}: {len(rainfall_events)} events to process")

        event_details_cache = {
            ev: get_rainfall_event_details(rainfall_events, ev)
            for ev in rainfall_events['event_num']}

        t0 = time.time()
        full_rain_cube = get_rainfall_cube_subsection(2015, ens_num, rainfall_cube_dir, 1, 2)
        FULL_MASK_2D   = mask_cube_with_catchment_full_grid(full_rain_cube[0], CATCHMENT_POLY, method='full_cell')

        year_cube, x_offset, y_offset = subset_cube_to_bbox(full_rain_cube, CATCHMENT_POLY, buffer=0)

        # Use year_cube's shape, not full_rain_cube's
        ny_sub = year_cube.shape[1]
        nx_sub = year_cube.shape[2]

        mask_2d_sub = FULL_MASK_2D[y_offset:y_offset + ny_sub, x_offset:x_offset + nx_sub]


        emit(f"EM{ens_num}: mask created in {time.time()-t0:.1f}s")
        del full_rain_cube

        for year, events_in_year in rainfall_events[:5].groupby('start_year'):
            print(f"Running for {year} for {ens_num}")
            for row in rainfall_events.itertuples():
                event_num     = int(row.event_num)
                event_details = event_details_cache[event_num]
                print(event_details)

                ###
                t0 = time.time()
                year_cube = get_rainfall_cube_subsection(year, ens_num, rainfall_cube_dir, 
                                                     event_details['start_idx'], event_details['stop_idx'])
                year_cube, x_offset, y_offset = subset_cube_to_bbox(year_cube, CATCHMENT_POLY, buffer=0)
                time_coord  = year_cube.coord('time')

                ####
                this_event_results = find_max_precip_location_new(
                    year_cube, event_details['start_idx'], event_details['stop_idx'],
                    x_offset=x_offset, y_offset=y_offset, mask_2d=mask_2d_sub)
                print(this_event_results)
                this_event_results['ens']       = ens_num
                this_event_results['start_idx'] = event_details['start_idx']
                this_event_results['stop_idx']  = event_details['stop_idx']
                this_event_results['max_precip_from_csv']  = event_details['max_precip_from_csv']

                t = time_coord.units.num2date(time_coord.points[this_event_results['t_local']])
                this_event_results['rainfall_peak_day'] = cftime.Datetime360Day(t.year, t.month, t.day)

                rainfall_at_peak  = get_data_at_peak_cell(year_cube, this_event_results, 'x_idx', 'y_idx')
                temp_profile_dict = find_temporal_profile_new(rainfall_at_peak, this_event_results, plot=True)
                this_event_results = {**this_event_results, **temp_profile_dict}

                results.append(this_event_results)
                results_this_ens.append(this_event_results)

                if math.isclose(this_event_results['max_precip'], this_event_results['max_precip_from_csv'], abs_tol=0.001):
                    found_mismatch = True
                    break
            if found_mismatch:
                    print("Houston we have a problemo")
                    break 

            emit(f"EM{ens_num} | year {year}: {len(events_in_year)} events in {time.time()-t0:.1f}s")
            del year_cube
            gc.collect()

        pd.DataFrame(results_this_ens).to_pickle(
            os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_EM{ens_num}.pkl"))
        emit(f"EM{ens_num}: saved")

    results_df  = pd.DataFrame(results)
    combined_fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}.pkl")
    results_df.to_pickle(combined_fp)
    emit(f"Combined file saved: {combined_fp}")

    for ens_num in ENSEMBLE_MEMBERS:
        fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_EM{ens_num}.pkl")
        if os.path.exists(fp):
            os.remove(fp)
    emit("Done — individual EM files cleaned up")

    return catchment_num


Catchment 81 (CreeGroup): missing ens ['01', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '15']
Catchment 23 (Tyne(Northumberland)) is complete
Catchment 77 (Esk(Dumfriesshire)) is complete
Catchment 82 (DoonGroup): missing ens ['01', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '15']
Catchment 68 (CheshireRiversGroup) is complete
Catchment 44 (FromeGroup): missing ens ['01', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '15']
Catchment 29 (AncholmeGroup): missing ens ['01', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '15']
Catchment 52 (SomersetRiversGroup): missing ens ['01', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '15']
Catchment 105 (InnerHebrides): missing ens ['05', '06', '07', '08', '09', '10', '11', '12', '13', '15']
Catchment 53 (Avon(Bristol)) is complete
Catchment 64 (DyfiGroup): missing ens ['01', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '15']
Catchment 13 (EskGroup) is comple

In [3]:
# catchment_num =23
# catchment_name  = CATCHMENT_LOOKUP_DICT[str(catchment_num)]
# rainfall_events_pkl = pd.read_pickle(f"../../Data/EventDetails/Catchment_{catchment_num}/{catchment_name}.pkl")
# rainfall_events_csv  = pd.read_csv(RAINFALL_CSV_DIR + f"{catchment_name}_{ens_num}_full_events_with_event_nums.csv")

# for i in range(0, len(rainfall_events_csv)):
#     print(i)
#     if not math.isclose(float(np.float32(rainfall_events_pkl['max_precip'].iloc[i])), 
#                                      float(np.float32(rainfall_events_csv['peaks'].iloc[i])),
#                                      abs_tol=0.001):
#                     print(f"MISMATCH — ens={ens_num}, year={year}, event={event_num}")
#                     print(f"  cube: {rainfall_events_pkl['max_precip'].iloc[i]:.8f}")
#                     print(f"  csv:  {rainfall_events_csv['peaks'].iloc[i]:.8f}")


In [4]:
class StopProcessing(Exception):
    pass

In [25]:
catchment_num =23
verbose=True

# ── Set up output routing ─────────────────────────────────────────────────
# When running in parallel, verbose=False routes all output to a per-catchment
# log file so workers don't interleave. When debugging in a notebook,
# verbose=True prints directly so you see output immediately.
log = setup_worker_logger(catchment_num)

def emit(msg):
    """Single call routes to print or log depending on verbose flag."""
    if verbose:
        print(msg)
    else:
        log.info(msg)

# Get data for the catchment
catchment_name  = CATCHMENT_LOOKUP_DICT[str(catchment_num)]
emit(f"Starting — {catchment_name}")
boundary_gdf    = CATCHMENTS[CATCHMENTS['HA_NUM'] == str(catchment_num)]
CATCHMENT_POLY = boundary_gdf.geometry.iloc[0]

# Create a list which will store all results (across ensemble members)
results = []

# ───────────────────────────────────────────────── 
# Create a mask for this catchment (over whole UK)
# Create a version of it over the area filtered to the catchment
# These are used later
# ───────────────────────────────────────────────── 
t0 = time.time()
full_rain_cube = get_rainfall_cube_subsection(2015, '01', f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/01/", 1, 2) #any year, any ensemble member
FULL_MASK_2D   = mask_cube_with_catchment_full_grid(full_rain_cube[0], CATCHMENT_POLY, method='center_point')
year_cube, mask_x_offset, mask_y_offset = subset_cube_to_bbox(full_rain_cube, CATCHMENT_POLY, buffer=0)
ny_sub = year_cube.shape[1]
nx_sub = year_cube.shape[2]
mask_2d_sub = FULL_MASK_2D[mask_y_offset:mask_y_offset + ny_sub, 
                            mask_x_offset:mask_x_offset + nx_sub]
emit(f"EM{ens_num}: mask created in {time.time()-t0:.1f}s")
del full_rain_cube, year_cube
        
# ─────────────────────────────────────────────────
# Get hydraulic conductivity data
# ─────────────────────────────────────────────────
HC_CUBE      = iris.load(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/HydraulicConductivity/5km_{catchment_num}.nc")[0]
HC_CUBE.data = np.where(FULL_MASK_2D, HC_CUBE.data, np.nan)
HC_CUBE = filter_closer_to_catchment(HC_CUBE, CATCHMENT_POLY, plot=False)
HC_DATA      = HC_CUBE.data  # realise once — shape (y, x)
flood_dir    = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/5km_total/Catchment_{catchment_num}/"

# ─────────────────────────────────────────────────
# HMM?
# ─────────────────────────────────────────────────
flood_numpy = {}  # will hold realised numpy arrays, keyed [depth]['area'/'vol']
flood_cubes = {}  # still needed for maybe_diagnose (expects iris cubes)
current_ens = None


# ───────────────────────────────────────────────── 
# Loop over EMs (wrap in try/pass to deal with errors from mismatch of max precip)
# ───────────────────────────────────────────────── 
try:
    for ens_num in ['01']:# ENSEMBLE_MEMBERS:
        start_time_ens = time.time()
        # Create a list which will store all results from this ensemble member
        results_this_ens = []
        
        # Check whether results for this ensemble member already exist, if so, add them to the list and move on        
        out_fp = f"{OUT_DIR}/Catchment_{catchment_num}/{catchment_name}_EM{ens_num}.pkl"
        if os.path.isfile(out_fp):
            # Read it in, and add to the dictionaries
            this_event_results = pd.read_pickle(out_fp)
            results.append(this_event_results)
            emit(f"EM{ens_num}: already exists, skipping")
            continue
        
        # Read in the csv with event details for this catchment/ensemble member
        rainfall_events  = pd.read_csv(RAINFALL_CSV_DIR + f"{catchment_name}_{ens_num}_full_events_with_event_nums.csv")
        rainfall_events['event_num'] = range(1, len(rainfall_events) + 1)
        # Specify filepath to the directory with netCDF for this ensemble member
        rainfall_cube_dir = f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/{ens_num}/"
        emit(f"EM{ens_num}: {len(rainfall_events)} events to process")
        
        # Create cached version of event details, for all events
        event_details_cache = {
            ev: get_rainfall_event_details(rainfall_events, ev)
            for ev in rainfall_events['event_num']}
        
        # ───────────────────────────────────────────────── 
        # Loop through each event
        # ───────────────────────────────────────────────── 
        for year, events_in_year in rainfall_events[:2].groupby('start_year'):
            print(f"Running for {year} for {ens_num}")
            for row in events_in_year.itertuples():
                # Get the event details from the cache
                event_num     = int(row.event_num)
                event_details = event_details_cache[event_num]

                # Get a portion of the cube, over the catchment, between the times when the maximum happened
                t0 = time.time()
                year_cube = get_rainfall_cube_subsection(year, ens_num, rainfall_cube_dir, 
                                                     event_details['start_idx'], event_details['stop_idx'])
                year_cube, x_offset, y_offset = subset_cube_to_bbox(year_cube, CATCHMENT_POLY, buffer=0)
                # Extract the times for this subset
                time_coord  = year_cube.coord('time')

                # Find the location of the max precipitation value
                # This masks out all cells not inside the catchment
                this_event_results = find_max_precip_location_new(
                    year_cube, event_details['start_idx'], event_details['stop_idx'],
                    x_offset=x_offset, y_offset=y_offset, mask_2d=mask_2d_sub)
                
                # Add key details to the results dictionary
                this_event_results['ens']       = ens_num
                this_event_results['start_idx'] = event_details['start_idx']
                this_event_results['stop_idx']  = event_details['stop_idx']
                this_event_results['max_precip_from_csv']  = event_details['max_precip_from_csv']
                
                # Get the day on which the peak occurred
                # This uses t_local to reference the times from the subset
                t = time_coord.units.num2date(time_coord.points[this_event_results['t_local']])
                this_event_results['rainfall_peak_day'] = cftime.Datetime360Day(t.year, t.month, t.day)
                
                # Get 1D cube, just at the location of the peak
                rainfall_at_peak  = get_data_at_peak_cell(year_cube, this_event_results, 'x_idx', 'y_idx')
                # Extract various temporal profile results
                temp_profile_dict = find_temporal_profile_new(rainfall_at_peak, this_event_results, plot=False)
                # Add these to dictionary of results 
                this_event_results = {**this_event_results, **temp_profile_dict}
                
                # Check whether the maximum precipitation matches the values in the input csv
                if not math.isclose(float(np.float32(this_event_results['max_precip'])), 
                                     float(np.float32(this_event_results['max_precip_from_csv'])),
                                     abs_tol=0.001):
                    print(f"MISMATCH — ens={ens_num}, year={year}, event={event_num}")
                    print(f"  cube: {this_event_results['max_precip']:.8f}")
                    print(f"  csv:  {this_event_results['max_precip_from_csv']:.8f}")
                    raise StopProcessing
                  
                # Add to both EM specific, and overall results
                results.append(this_event_results)
                results_this_ens.append(this_event_results)
                
            emit(f"EM{ens_num} | year {year}: {len(events_in_year)} events in {time.time()-t0:.1f}s")
            del year_cube
            gc.collect()

        # Save results for this ensemble member to file
#         pd.DataFrame(results_this_ens).to_pickle(
#             os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_EM{ens_num}.pkl"))
        emit(f"EM{ens_num}: saved in {round(time.time() - start_time_ens,2)} seconds")
    
except StopProcessing:
    print("Stopped at first mismatch — results not saved.")
    
else:
    # Only runs if try completed without exception
    results_df  = pd.DataFrame(results)
    
#     combined_fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}.pkl")
#     results_df.to_pickle(combined_fp)
#     emit(f"Combined file saved: {combined_fp}")
#     for ens_num in ENSEMBLE_MEMBERS:
#         fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_EM{ens_num}.pkl")
#         if os.path.exists(fp):
#             os.remove(fp)
#     emit("Done — individual EM files cleaned up")

Starting — Tyne(Northumberland)
EM01: mask created in 0.3s
EM01: 95 events to process
Running for 1992 for 01
EM01 | year 1992: 1 events in 1.6s
Running for 1994 for 01
EM01 | year 1994: 1 events in 1.6s
EM01: saved in 3.72


,start_indices,stop_indices,event_durations,peaks,start_year,start_month,start_day,start_hour,start_time,stop_time,start_time_seconds,stop_time_seconds,event_vol,max_extent,acc_total,acc_3hr,acc_6hr,acc_12hr,acc_24hr,event_num
0,5604,5616,12,33.70412,1991,7,24,11,186323.5,186335.5,670764600,670807800,395.739684,18,395.739684,279.40766,393.515020,395.731279,NaN,1
1,5557,5568,11,36.13153,1993,7,22,12,203556.5,203567.5,732803400,732843000,537.017821,24,537.017821,418.93990,532.018370,537.017821,NaN,2
2,5602,5618,16,33.17259,1993,7,24,9,203601.5,203617.5,732965400,733023000,973.625061,30,973.625061,712.31010,877.482960,973.621875,NaN,3
3,5734,5765,31,47.58703,1995,7,29,21,221013.5,221044.5,795648600,795760200,1671.167858,28,1671.167858,383.76441,751.413900,1222.365350,1640.973394,4
4,3851,3870,19,31.52430,1998,5,11,10,245050.5,245069.5,882181800,882250200,1343.188339,33,1343.188339,552.71930,879.082470,1245.029775,NaN,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,5884,5907,23,36.47309,2077,8,6,3,929643.5,929666.5,3346716600,3346799400,1482.101702,38,1482.101702,657.98050,1084.244300,1431.882148,1482.101702,83
83,5989,6001,12,33.72202,2078,8,10,12,938388.5,938400.5,3378198600,3378241800,526.927571,21,526.927571,304.16111,526.742750,526.927320,NaN,84
84,6292,6312,20,33.25567,2078,8,23,3,938691.5,938711.5,3379289400,3379361400,1440.452670,63,1440.452670,606.85396,943.636600,1331.866338,NaN,85
85,3208,3228,20,32.46070,2080,4,14,15,952887.5,952907.5,3430395000,3430467000,1643.681949,82,1643.681949,955.93720,1157.366812,1488.003194,NaN,86


In [6]:
catchment_num =23
verbose=True

# ── Set up output routing ─────────────────────────────────────────────────
# When running in parallel, verbose=False routes all output to a per-catchment
# log file so workers don't interleave. When debugging in a notebook,
# verbose=True prints directly so you see output immediately.
log = setup_worker_logger(catchment_num)

def emit(msg):
    """Single call routes to print or log depending on verbose flag."""
    if verbose:
        print(msg)
    else:
        log.info(msg)

catchment_name  = CATCHMENT_LOOKUP_DICT[str(catchment_num)]
emit(f"Starting — {catchment_name}")
boundary_gdf    = CATCHMENTS[CATCHMENTS['HA_NUM'] == str(catchment_num)]
CATCHMENT_POLY = boundary_gdf.geometry.iloc[0]
results         = []

try:
    for ens_num in ENSEMBLE_MEMBERS:
        out_fp = f"{OUT_DIR}/Catchment_{catchment_num}/{catchment_name}_EM{ens_num}.pkl"
        if os.path.isfile(out_fp):
            emit(f"EM{ens_num}: already exists, skipping")
            continue

        results_this_ens = []
        rainfall_events  = pd.read_csv(RAINFALL_CSV_DIR + f"{catchment_name}_{ens_num}_full_events_with_event_nums.csv")
        rainfall_events['event_num'] = range(1, len(rainfall_events) + 1)
        rainfall_cube_dir = f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/{ens_num}/"
        emit(f"EM{ens_num}: {len(rainfall_events)} events to process")

        event_details_cache = {
            ev: get_rainfall_event_details(rainfall_events, ev)
            for ev in rainfall_events['event_num']}

        t0 = time.time()
        full_rain_cube = get_rainfall_cube_subsection(2015, ens_num, rainfall_cube_dir, 1, 2)
        FULL_MASK_2D   = mask_cube_with_catchment_full_grid(full_rain_cube[0], CATCHMENT_POLY, method='center_point')

        year_cube, x_offset, y_offset = subset_cube_to_bbox(full_rain_cube, CATCHMENT_POLY, buffer=0)

        # Use year_cube's shape, not full_rain_cube's
        ny_sub = year_cube.shape[1]
        nx_sub = year_cube.shape[2]

        mask_2d_sub = FULL_MASK_2D[y_offset:y_offset + ny_sub, x_offset:x_offset + nx_sub]

        emit(f"EM{ens_num}: mask created in {time.time()-t0:.1f}s")
        del full_rain_cube

        for year, events_in_year in rainfall_events.groupby('start_year'):
            print(f"Running for {year} for {ens_num}")
            for row in events_in_year.itertuples():
                event_num     = int(row.event_num)
                event_details = event_details_cache[event_num]

                ###
                t0 = time.time()
                year_cube = get_rainfall_cube_subsection(year, ens_num, rainfall_cube_dir, 
                                                     event_details['start_idx'], event_details['stop_idx'])
                year_cube, x_offset, y_offset = subset_cube_to_bbox(year_cube, _CATCHMENT_POLY, buffer=0)
                time_coord  = year_cube.coord('time')

                ####
                this_event_results = find_max_precip_location_new(
                    year_cube, event_details['start_idx'], event_details['stop_idx'],
                    x_offset=x_offset, y_offset=y_offset, mask_2d=mask_2d_sub)
                this_event_results['ens']       = ens_num
                this_event_results['start_idx'] = event_details['start_idx']
                this_event_results['stop_idx']  = event_details['stop_idx']
                this_event_results['max_precip_from_csv']  = event_details['max_precip_from_csv']

                t = time_coord.units.num2date(time_coord.points[this_event_results['t_local']])
                this_event_results['rainfall_peak_day'] = cftime.Datetime360Day(t.year, t.month, t.day)

                rainfall_at_peak  = get_data_at_peak_cell(year_cube, this_event_results, 'x_idx', 'y_idx')
                temp_profile_dict = find_temporal_profile_new(rainfall_at_peak, this_event_results, plot=False)
                this_event_results = {**this_event_results, **temp_profile_dict}

                results.append(this_event_results)
                results_this_ens.append(this_event_results)
                if not math.isclose(float(np.float32(this_event_results['max_precip'])), 
                                     float(np.float32(this_event_results['max_precip_from_csv'])),
                                     abs_tol=0.001):
                    print(f"MISMATCH — ens={ens_num}, year={year}, event={event_num}")
                    print(f"  cube: {this_event_results['max_precip']:.8f}")
                    print(f"  csv:  {this_event_results['max_precip_from_csv']:.8f}")
                    raise StopProcessing
            emit(f"EM{ens_num} | year {year}: {len(events_in_year)} events in {time.time()-t0:.1f}s")
            del year_cube
            gc.collect()
except StopProcessing:
    print("Stopped at first mismatch.")
    
#     pd.DataFrame(results_this_ens).to_pickle(
#         os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_EM{ens_num}.pkl"))
#     emit(f"EM{ens_num}: saved")

# results_df  = pd.DataFrame(results)
# combined_fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}.pkl")
# results_df.to_pickle(combined_fp)
# emit(f"Combined file saved: {combined_fp}")

# for ens_num in ENSEMBLE_MEMBERS:
#     fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_EM{ens_num}.pkl")
#     if os.path.exists(fp):
#         os.remove(fp)
# emit("Done — individual EM files cleaned up")

Starting — Tyne(Northumberland)
EM01: 95 events to process
EM01: mask created in 0.3s
Running for 1992 for 01
EM01 | year 1992: 1 events in 1.6s
Running for 1994 for 01
EM01 | year 1994: 2 events in 1.6s
Running for 1995 for 01
EM01 | year 1995: 1 events in 1.7s
Running for 2001 for 01



KeyboardInterrupt



In [ ]:
year_cube = get_rainfall_cube_subsection(year, ens_num, rainfall_cube_dir, 
                                     event_details['start_idx'], event_details['stop_idx'])
year_cube, x_offset, y_offset = subset_cube_to_bbox(year_cube, _CATCHMENT_POLY, buffer=0)
np.nanmax(year_cube.data)
for i in range(1,19):
    test = year_cube[i,:,:]
    print(np.nanmax(test.data))
iplt.pcolormesh(year_cube[1,:,:])

In [ ]:
# def emit(msg):
#     """Single call routes to print or log depending on verbose flag."""
#     if verbose:
#         print(msg)
#     else:
#         log.info(msg)
# verbose=True

# # ── Set up output routing ─────────────────────────────────────────────────
# # When running in parallel, verbose=False routes all output to a per-catchment
# # log file so workers don't interleave. When debugging in a notebook,
# # verbose=True prints directly so you see output immediately.
# log = setup_worker_logger(catchment_num)

# catchment_name  = CATCHMENT_LOOKUP_DICT[str(catchment_num)]
# emit(f"Starting — {catchment_name}")
# boundary_gdf    = CATCHMENTS[CATCHMENTS['HA_NUM'] == str(catchment_num)]
# _CATCHMENT_POLY = boundary_gdf.geometry.iloc[0]
# results         = []

# ens_num = '01'

# out_fp = f"{OUT_DIR}/Catchment_{catchment_num}/{catchment_name}_EM{ens_num}.pkl"
# if os.path.isfile(out_fp):
#     print(f"EM{ens_num}: already exists, skipping")


# results_this_ens = []
# rainfall_events  = pd.read_csv(RAINFALL_CSV_DIR + f"{catchment_name}_{ens_num}_full_events_with_event_nums.csv")
# rainfall_events['event_num'] = range(1, len(rainfall_events) + 1)
# rainfall_cube_dir = f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/{ens_num}/"
# print(f"EM{ens_num}: {len(rainfall_events)} events to process")

# event_details_cache = {
#     ev: get_rainfall_event_details(rainfall_events, ev)
#     for ev in rainfall_events['event_num']}

# t0 = time.time()
# full_rain_cube = get_rainfall_cube_subsection(2015, ens_num, rainfall_cube_dir,1,2)
# FULL_MASK_2D   = mask_cube_with_catchment_full_grid(full_rain_cube[0], _CATCHMENT_POLY, method='full_cell')
# nx_sub      = full_rain_cube.shape[2]
# ny_sub      = full_rain_cube.shape[1]
# print(nx_sub, ny_sub)
# year_cube, x_offset, y_offset = subset_cube_to_bbox(full_rain_cube, _CATCHMENT_POLY, buffer=0)
# mask_2d_sub = FULL_MASK_2D[y_offset:y_offset+ny_sub, x_offset:x_offset+nx_sub]
# time_coord  = year_cube.coord('time')
# emit(f"EM{ens_num}: mask created in {time.time()-t0:.1f}s")
# # del full_rain_cube


In [ ]:
results_this_ens = []
rainfall_events  = pd.read_csv(RAINFALL_CSV_DIR + f"{catchment_name}_{ens_num}_full_events_with_event_nums.csv")
rainfall_events['event_num'] = range(1, len(rainfall_events) + 1)
rainfall_cube_dir = f"/scratch/hydro5/users/ld14116/SDM_bias_correction/Hourly/{ens_num}/"
emit(f"EM{ens_num}: {len(rainfall_events)} events to process")

event_details_cache = {
    ev: get_rainfall_event_details(rainfall_events, ev)
    for ev in rainfall_events['event_num']}

t0 = time.time()
full_rain_cube = get_rainfall_cube_subsection(2015, ens_num, rainfall_cube_dir, 1, 2)
FULL_MASK_2D   = mask_cube_with_catchment_full_grid(full_rain_cube[0], CATCHMENT_POLY, method='full_cell')

year_cube, x_offset, y_offset = subset_cube_to_bbox(full_rain_cube, CATCHMENT_POLY, buffer=0)

# Use year_cube's shape, not full_rain_cube's
ny_sub = year_cube.shape[1]
nx_sub = year_cube.shape[2]

mask_2d_sub = FULL_MASK_2D[y_offset:y_offset + ny_sub, x_offset:x_offset + nx_sub]


emit(f"EM{ens_num}: mask created in {time.time()-t0:.1f}s")
del full_rain_cube

for year, events_in_year in rainfall_events[:5].groupby('start_year'):
    print(f"Running for {year} for {ens_num}")
    for row in events_in_year.itertuples():
        event_num     = int(row.event_num)
        event_details = event_details_cache[event_num]
        print(event_details)
        
        ###
        t0 = time.time()
        year_cube = get_rainfall_cube_subsection(year, ens_num, rainfall_cube_dir, 
                                             event_details['start_idx'], event_details['stop_idx'])
        year_cube, x_offset, y_offset = subset_cube_to_bbox(year_cube, CATCHMENT_POLY, buffer=0)
        time_coord  = year_cube.coord('time')
        
        ####
        this_event_results = find_max_precip_location_new(
            year_cube, event_details['start_idx'], event_details['stop_idx'],
            x_offset=x_offset, y_offset=y_offset, mask_2d=mask_2d_sub)
        print(this_event_results)
        this_event_results['ens']       = ens_num
        this_event_results['start_idx'] = event_details['start_idx']
        this_event_results['stop_idx']  = event_details['stop_idx']
        this_event_results['max_precip_from_csv']  = event_details['max_precip_from_csv']

        t = time_coord.units.num2date(time_coord.points[this_event_results['t_local']])
        this_event_results['rainfall_peak_day'] = cftime.Datetime360Day(t.year, t.month, t.day)

        rainfall_at_peak  = get_data_at_peak_cell(year_cube, this_event_results, 'x_idx', 'y_idx')
        temp_profile_dict = find_temporal_profile_new(rainfall_at_peak, this_event_results, plot=True)
        this_event_results = {**this_event_results, **temp_profile_dict}

        results.append(this_event_results)
        results_this_ens.append(this_event_results)
        
        if math.isclose(this_event_results['max_precip'], this_event_results['max_precip_from_csv'], abs_tol=0.001):
            print("Houston we have a problemo")

    emit(f"EM{ens_num} | year {year}: {len(events_in_year)} events in {time.time()-t0:.1f}s")
    del year_cube
    gc.collect()

# pd.DataFrame(results_this_ens).to_pickle(
#     os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_EM{ens_num}.pkl"))
# emit(f"EM{ens_num}: saved")

# results_df  = pd.DataFrame(results)
# combined_fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}.pkl")
# results_df.to_pickle(combined_fp)
# emit(f"Combined file saved: {combined_fp}")

# for ens_num in ENSEMBLE_MEMBERS:
#     fp = os.path.join(OUT_DIR, f"Catchment_{catchment_num}", f"{catchment_name}_EM{ens_num}.pkl")
#     if os.path.exists(fp):
#         os.remove(fp)
# emit("Done — individual EM files cleaned up")